In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests
from pydantic import BaseModel, ConfigDict, Field, ValidationError


In [2]:
url = (
    r"https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.jsonlKVP"
    r"?CatalogNumber,ScientificName,latitude,longitude,DepthInMeters,LargeMarineEcosystem,ObservationDate,EventID,SurveyID,AphiaID,Genus,Family,Order,IndividualCount,Density,Temperature,Salinity,Oxygen"
    r"&ShallowFlag=0"
    r"&ObservationYear%3E=2018"
    r"&Country=%22USA%22"
    r"&LargeMarineEcosystem=~%22%28Southeast%20U.S.%20Continental%20Shelf%7CGulf%20of%20Mexico%7CCaribbean%20Sea%29%22"
)

print(url)

https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.jsonlKVP?CatalogNumber,ScientificName,latitude,longitude,DepthInMeters,LargeMarineEcosystem,ObservationDate,EventID,SurveyID,AphiaID,Genus,Family,Order,IndividualCount,Density,Temperature,Salinity,Oxygen&ShallowFlag=0&ObservationYear%3E=2018&Country=%22USA%22&LargeMarineEcosystem=~%22%28Southeast%20U.S.%20Continental%20Shelf%7CGulf%20of%20Mexico%7CCaribbean%20Sea%29%22


In [3]:
class DeepSeaCoralRecord(BaseModel):
    model_config = ConfigDict(populate_by_name=True, frozen=True)

    latitude: float
    longitude: float
    catalog_num: int = Field(alias="CatalogNumber")
    scientific_name: str | None = Field(default=None, alias="ScientificName")
    depth_m: float | None = Field(default=None, alias="DepthInMeters")
    large_marine_ecosystem: str = Field(alias="LargeMarineEcosystem")
    observation_date: str = Field(alias="ObservationDate")
    event_id: str = Field(alias="EventID")
    survey_id: str = Field(alias="SurveyID")
    aphia_id: int | None = Field(default=None, alias="AphiaID")
    genus: str | None = Field(default=None, alias="Genus")
    family: str | None = Field(default=None, alias="Family")
    order: str | None = Field(default=None, alias="Order")
    individual_count: int | None = Field(default=None, alias="IndividualCount")
    density: float | None = Field(default=None, alias="Density")
    temperature: float | None = Field(default=None, alias="Temperature")
    salinity: float | None = Field(default=None, alias="Salinity")
    oxygen: float | None = Field(default=None, alias="Oxygen")


In [4]:
def download(url: str, dest: Path) -> None:
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            f.writelines(r.iter_content(1 << 20))

In [ ]:
def read_records(
    file_path: Path,
) -> tuple[list[DeepSeaCoralRecord], list[dict[str, str]]]:
    records: list[DeepSeaCoralRecord] = []
    errors: list[dict[str, str]] = []
    with open(file_path) as f:
        for line in f:
            try:
                record = DeepSeaCoralRecord(**json.loads(line))
                records.append(record)
            except ValidationError as e:
                errors.append({"line": line, "error": str(e)})
    return records, errors

In [6]:
dest = Path("../data/raw/deep_sea_corals_se.jsonl")
dest.parent.mkdir(exist_ok=True)

download(url, dest)
records, errors = read_records(dest)

print(f"Validated: {len(records)} records")
print(f"Errors: {len(errors)} records")

Validated: 84756 records
Errors: 0 records


In [7]:
df = pd.DataFrame(record.model_dump() for record in records)
df.head(10)

,latitude,longitude,catalog_num,scientific_name,depth_m,large_marine_ecosystem,observation_date,event_id,survey_id,aphia_id,genus,family,order,individual_count,density,temperature,salinity,oxygen
0,17.92201,-66.64733,953318,Pennatuloidea,284.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,2.0,NaN,NaN,NaN,NaN
1,17.92225,-66.64734,953319,Pennatuloidea,282.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
2,17.92228,-66.64734,953320,Pennatuloidea,280.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
3,17.92228,-66.64732,953321,Pennatuloidea,279.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
4,17.92227,-66.64732,953322,Pennatuloidea,279.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
5,17.92231,-66.64733,953323,Pennatuloidea,279.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
6,17.92244,-66.64727,953324,Pennatuloidea,278.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,2.0,NaN,NaN,NaN,NaN
7,17.92242,-66.64729,953325,Pennatuloidea,277.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
8,17.92252,-66.64732,953326,Pennatuloidea,275.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
9,17.92255,-66.64732,953327,Pennatuloidea,274.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,2.0,NaN,NaN,NaN,NaN


,latitude,longitude,catalog_num,scientific_name,depth_m,large_marine_ecosystem,observation_date,event_id,survey_id,aphia_id,genus,family,order,individual_count,density,temperature,salinity,oxygen
0,17.92201,-66.64733,953318,Pennatuloidea,284.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,2.0,NaN,NaN,NaN,NaN
1,17.92225,-66.64734,953319,Pennatuloidea,282.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
2,17.92228,-66.64734,953320,Pennatuloidea,280.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
3,17.92228,-66.64732,953321,Pennatuloidea,279.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
4,17.92227,-66.64732,953322,Pennatuloidea,279.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
5,17.92231,-66.64733,953323,Pennatuloidea,279.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
6,17.92244,-66.64727,953324,Pennatuloidea,278.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,2.0,NaN,NaN,NaN,NaN
7,17.92242,-66.64729,953325,Pennatuloidea,277.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
8,17.92252,-66.64732,953326,Pennatuloidea,275.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,1.0,NaN,NaN,NaN,NaN
9,17.92255,-66.64732,953327,Pennatuloidea,274.0,Caribbean Sea,2018-06-30,NF1804_01,NF1804,1609360,,,Scleralcyonacea,2.0,NaN,NaN,NaN,NaN


In [8]:
df.to_parquet("../data/interim/se_coral_records.parquet")